# 18. LightGBM Gain-Based Feature Interpretation — Facial Skincare

This notebook interprets the RankP LightGBM models fitted on the query-only candidate pool. It first reproduces stored native scores using each fold model only on its assigned held-out rows. Strict-cold rows that use the Base fallback are excluded from this native-model reproduction check.

The feature interpretation itself uses normalised LightGBM gain extracted from the fitted fold boosters and aggregated under the fixed feature-group registry. It reports fold-level stability, cross-fold rank correlations, and the grouped gain shares at candidate depth 1,000 used for Figure 7.1. Previously-reviewed-item diagnostic variables are verified as absent from the interpreted model registry.

Gain importance describes how the fitted trees used the registered variables; it is not a causal contribution estimate and may favour features with more available split points. No held-out permutation importance or SHAP estimate is produced by this notebook.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install -U lightgbm scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 50.2 MB/s eta 0:00:00


In [3]:
# ==== Imports, Identity, and Contract Paths ====
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import lightgbm as lgb


NOTEBOOK_NAME = "18_lightgbm_heldout_interpretation_face.ipynb"
CATEGORY_ID = "face"
CATEGORY_KEY = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
MODEL_FAMILY = "lightgbm"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
MODEL_ARTIFACT_DIR = PROJECT_ROOT / "outputs" / "stage2_nonpersonalized_rerank" / "lightgbm"
INTERPRETATION_MANIFEST_PATH = MODEL_ARTIFACT_DIR / "feature_interpretation_manifest.json"
CANONICAL_RAW_PATH = PROJECT_ROOT / "outputs" / "pipeline_aggregate" / "pipeline_canonical_per_case_metrics.parquet"

OUT_DIR = PROJECT_ROOT / "outputs" / "analysis" / "lightgbm_interpretation"
if OUT_DIR.parent != PROJECT_ROOT / "outputs" / "analysis":
    raise RuntimeError("LightGBM interpretation output directory must remain under category analysis outputs.")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20250322
PRIMARY_K = 5
PRIMARY_POOL_DEPTH = 1000
REPRODUCTION_ATOL = 1e-5
NATIVE_PREDICTION_SOURCE = "prior_aware_native"
FALLBACK_PREDICTION_SOURCE = "p2q_cold_fallback"

LEGACY_INTERPRETATION_GROUPS = [
    "query_item", "retrieval", "candidate_item", "history_depth_recency",
    "functional_facet_preference", "brand_affinity",
]
REQUIRED_SEMANTIC_GROUPS = [
    "retrieval_rank",
    "history_depth_recency",
    "functional_preference",
    "query_profile_interaction",
    "candidate_history_interaction",
    "brand_prior",
]

OUTPUT_FILES = {
    "gain_by_fold": OUT_DIR / "lightgbm_gain_importance_by_fold.csv",
    "gain_stability": OUT_DIR / "lightgbm_gain_importance_stability.csv",
    "fold_rank_correlation": OUT_DIR / "lightgbm_gain_fold_rank_correlation.csv",
    "feature_group_gain_summary": OUT_DIR / "lightgbm_feature_group_gain_summary.csv",
    "primary_feature_group_gain_summary": OUT_DIR / "lightgbm_primary_feature_group_gain_summary_pool1000.csv",
    "qc": OUT_DIR / "lightgbm_interpretation_qc.csv",
    "manifest": OUT_DIR / "run_manifest.json",
}


In [4]:
# ==== Validation Helpers ====
def require_columns(frame, required, label):
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(frame, label):
    duplicated = frame.columns[frame.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicated columns: {duplicated}")


def facet_feature_role(feature):
    if feature.startswith("qmatch__"):
        return "query_candidate_match"
    if feature.startswith("uaff__"):
        return "candidate_profile_affinity"
    if feature.startswith("ucountlog__"):
        return "candidate_profile_count"
    if feature.endswith("_recency_days"):
        return "profile_recency"
    if feature.endswith("_recent_count_180d"):
        return "profile_recent_count"
    return "other"


In [5]:
# ==== Constants Recovery Guard (Out-of-order Run Protection) ====
# Recover the Notebook 18 constants when this cell is run out of order.
if "INTERPRETATION_MANIFEST_PATH" not in globals() or "CANONICAL_RAW_PATH" not in globals() or "OUT_DIR" not in globals():
    from pathlib import Path
    import json
    import pandas as pd

    NOTEBOOK_NAME = globals().get("NOTEBOOK_NAME", "18_lightgbm_heldout_interpretation_face.ipynb")
    CATEGORY_ID = globals().get("CATEGORY_ID", "face")
    CATEGORY_KEY = globals().get("CATEGORY_KEY", "facial_skincare")
    CATEGORY_LABEL = globals().get("CATEGORY_LABEL", "Facial Skincare")
    MODEL_FAMILY = globals().get("MODEL_FAMILY", "lightgbm")
    PROJECT_ROOT = Path(globals().get(
        "PROJECT_ROOT",
        f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}",
    ))
    MODEL_ARTIFACT_DIR = PROJECT_ROOT / "outputs" / "stage2_nonpersonalized_rerank" / "lightgbm"
    INTERPRETATION_MANIFEST_PATH = MODEL_ARTIFACT_DIR / "feature_interpretation_manifest.json"
    CANONICAL_RAW_PATH = PROJECT_ROOT / "outputs" / "pipeline_aggregate" / "pipeline_canonical_per_case_metrics.parquet"
    OUT_DIR = PROJECT_ROOT / "outputs" / "analysis" / "lightgbm_interpretation"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    PRIMARY_K = globals().get("PRIMARY_K", 5)
    PRIMARY_POOL_DEPTH = globals().get("PRIMARY_POOL_DEPTH", 1000)
    REPRODUCTION_ATOL = globals().get("REPRODUCTION_ATOL", 1e-5)
    NATIVE_PREDICTION_SOURCE = globals().get("NATIVE_PREDICTION_SOURCE", "prior_aware_native")
    FALLBACK_PREDICTION_SOURCE = "p2q_cold_fallback"
    LEGACY_INTERPRETATION_GROUPS = globals().get("LEGACY_INTERPRETATION_GROUPS", [
        "query_item", "retrieval", "candidate_item", "history_depth_recency",
        "functional_facet_preference", "brand_affinity",
    ])
    REQUIRED_SEMANTIC_GROUPS = globals().get("REQUIRED_SEMANTIC_GROUPS", [
        "retrieval_rank", "history_depth_recency", "functional_preference",
        "query_profile_interaction", "candidate_history_interaction", "brand_prior",
    ])
    OUTPUT_FILES = globals().get("OUTPUT_FILES", {
        "gain_by_fold": OUT_DIR / "lightgbm_gain_importance_by_fold.csv",
        "gain_stability": OUT_DIR / "lightgbm_gain_importance_stability.csv",
        "fold_rank_correlation": OUT_DIR / "lightgbm_gain_fold_rank_correlation.csv",
        "feature_group_gain_summary": OUT_DIR / "lightgbm_feature_group_gain_summary.csv",
        "primary_feature_group_gain_summary": OUT_DIR / "lightgbm_primary_feature_group_gain_summary_pool1000.csv",
        "qc": OUT_DIR / "lightgbm_interpretation_qc.csv",
        "manifest": OUT_DIR / "run_manifest.json",
    })

# The current Notebook 13b fallback source token is p2q_cold_fallback. Normalize
# stale in-kernel constants when this validation cell is rerun out of order.
FALLBACK_PREDICTION_SOURCE = "p2q_cold_fallback"

if not INTERPRETATION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing LightGBM interpretation manifest: {INTERPRETATION_MANIFEST_PATH}")
if not CANONICAL_RAW_PATH.exists():
    raise FileNotFoundError(f"Missing Notebook 14 canonical raw artifact: {CANONICAL_RAW_PATH}")

interpretation_manifest = json.loads(INTERPRETATION_MANIFEST_PATH.read_text(encoding="utf-8"))
if interpretation_manifest.get("model_family") != MODEL_FAMILY:
    raise RuntimeError("Interpretation manifest is not a LightGBM contract.")
if interpretation_manifest.get("category_id") != CATEGORY_ID:
    raise RuntimeError(
        f"Category mismatch: expected={CATEGORY_ID}, manifest={interpretation_manifest.get('category_id')}"
    )
condition_name = str(interpretation_manifest.get("condition_name"))
model_source_condition = str(interpretation_manifest.get("model_source_condition", condition_name))
if condition_name != "P2-P":
    raise RuntimeError(f"Notebook 18 requires the RankP All Prior contract, found {condition_name!r}.")
if model_source_condition != "P2-P":
    raise RuntimeError(f"Fold-model source must be RankP/P2-P, found {model_source_condition!r}.")

feature_names = list(interpretation_manifest.get("feature_names", []))
if not feature_names or len(feature_names) != len(set(feature_names)):
    raise RuntimeError("LightGBM feature names are empty or duplicated.")
EXACT_ITEM_FAMILIARITY_FEATURES = {
    "user_item_seen_strength",
    "user_item_recency_days",
    "user_item_recent_count_180d",
}
exact_item_in_primary = sorted(set(feature_names).intersection(EXACT_ITEM_FAMILIARITY_FEATURES))
if exact_item_in_primary:
    raise RuntimeError(
        "Previously-reviewed-item diagnostics entered the primary LightGBM interpretation input: "
        f"{exact_item_in_primary}"
    )
if interpretation_manifest.get("specification_role", "primary") != "primary":
    raise RuntimeError("Primary LightGBM interpretation requires specification_role=primary.")
if bool(interpretation_manifest.get("exact_item_familiarity_enabled", False)):
    raise RuntimeError("Primary LightGBM interpretation cannot enable exact-item familiarity.")
expected_feature_dtypes = {feature: "float32" for feature in feature_names}
if interpretation_manifest.get("feature_dtypes") != expected_feature_dtypes:
    raise RuntimeError("Interpretation manifest feature dtypes differ from the executed float32 contract.")

candidate_source_contract = interpretation_manifest.get("candidate_source_contract", {})
required_candidate_contract = {
    "candidate_source_output_key": "query_only_winner_long",
    "candidate_scores_preserved": True,
    "retrieval_recomputed": False,
    "complete_case_universe_preserved": True,
}
for key, expected_value in required_candidate_contract.items():
    if candidate_source_contract.get(key) != expected_value:
        raise RuntimeError(
            f"Candidate-source contract mismatch for {key}: "
            f"expected={expected_value!r}, actual={candidate_source_contract.get(key)!r}"
        )

feature_mapping_path = Path(interpretation_manifest["feature_group_mapping_path"])
feature_mapping_df = pd.read_csv(feature_mapping_path)
require_columns(
    feature_mapping_df,
    ["feature_position", "feature", "dtype", "interpretation_group"],
    "feature-group mapping",
)
feature_mapping_df = feature_mapping_df.sort_values("feature_position", kind="mergesort").reset_index(drop=True)
if feature_mapping_df["feature"].astype(str).tolist() != feature_names:
    raise RuntimeError("Feature-group mapping order differs from the model feature order.")
if not feature_mapping_df["dtype"].astype(str).eq("float32").all():
    raise RuntimeError("Feature-group mapping dtypes differ from the executed float32 contract.")
unexpected_groups = sorted(
    set(feature_mapping_df["interpretation_group"].astype(str)).difference(LEGACY_INTERPRETATION_GROUPS)
)
if unexpected_groups:
    raise RuntimeError(f"Unresolved LightGBM interpretation groups: {unexpected_groups}")
feature_to_group = feature_mapping_df.set_index("feature")["interpretation_group"].astype(str).to_dict()
group_to_features = {
    group: feature_mapping_df.loc[
        feature_mapping_df["interpretation_group"].astype(str).eq(group), "feature"
    ].astype(str).tolist()
    for group in LEGACY_INTERPRETATION_GROUPS
}
if not group_to_features["brand_affinity"]:
    raise RuntimeError("Brand-affinity features must be present and reported separately for RankP.")

forbidden_feature_names = {
    "timestamp_ms", "target_timestamp_ms", "case_id", "query_id", "user_id",
    "target_parent_asin", "candidate_parent_asin", "target_item_id", "candidate_item_id",
    "gt_item_id", "is_gt", "label", "target_rank_desc", "query_text", "raw_query_text",
    "review_text", "review_body", "raw_review_text", "target_review_text", "heldout_review_text",
}
forbidden_feature_prefixes = (
    "target_", "heldout_", "future_", "post_target_", "posttarget_",
    "raw_review_", "review_text", "review_body",
)
leakage_features = sorted(
    feature for feature in feature_names
    if feature in forbidden_feature_names or feature.startswith(forbidden_feature_prefixes)
)
brand_query_features = sorted(
    feature for feature in feature_names
    if feature.startswith("qmatch__brand") or "query_brand" in feature
)
if leakage_features or brand_query_features:
    raise RuntimeError(
        f"Target/future or query-brand leakage features detected: {leakage_features + brand_query_features}"
    )

brand_contract = interpretation_manifest.get("brand_all_prior_feature_contract_flags", {})
brand_contract_features = [
    feature for feature in brand_contract.get("brand_affinity_feature_columns", [])
    if feature in feature_names
]
brand_prior_features = [
    feature for feature in brand_contract_features if feature != "candidate_brand_present"
]

def is_history_depth_or_recency(feature):
    if feature in EXACT_ITEM_FAMILIARITY_FEATURES:
        return False
    if feature in {"prior_review_n_log1p", "prior_item_n_log1p"}:
        return True
    if feature in brand_contract_features:
        return False
    return feature.startswith("user_") and feature.endswith(
        ("_recency_days", "_recent_count_180d", "_interaction_gap_days")
    )


semantic_group_to_features = {
    "retrieval_rank": list(group_to_features["retrieval"]),
    "history_depth_recency": [
        feature for feature in feature_names if is_history_depth_or_recency(feature)
    ],
    "functional_preference": [
        feature for feature in feature_names
        if feature.startswith("uaff__")
        or feature in {"user_entropy_norm_mean", "user_top_share_mean", "user_item_affinity"}
    ],
    "query_profile_interaction": [
        feature for feature in feature_names
        if feature.startswith(("qprofile__", "query_profile__", "query_history__", "qhist__"))
    ],
    "candidate_history_interaction": [
        feature for feature in feature_names
        if feature.startswith(("ucountlog__", "useen__"))
    ],
    "brand_prior": brand_prior_features,
}
semantic_membership = [
    (semantic_group, feature)
    for semantic_group, features in semantic_group_to_features.items()
    for feature in features
]
duplicated_semantic_features = sorted(
    feature for feature in {feature for _, feature in semantic_membership}
    if sum(member_feature == feature for _, member_feature in semantic_membership) > 1
)
if duplicated_semantic_features:
    raise RuntimeError(f"Semantic groups must be non-overlapping: {duplicated_semantic_features}")
for semantic_group in set(REQUIRED_SEMANTIC_GROUPS).difference({"query_profile_interaction"}):
    if not semantic_group_to_features[semantic_group]:
        raise RuntimeError(f"Required semantic group has no executed features: {semantic_group}")

semantic_group_definitions = {
    "retrieval_rank": "Stage-1 candidate rank and normalized retrieval-score features.",
    "history_depth_recency": "Leakage-safe user-history quantity and user/facet recency features.",
    "functional_preference": "Functional candidate-profile affinity and profile-dispersion features.",
    "query_profile_interaction": "Direct executed query-profile interaction features only; no proxy substitution.",
    "candidate_history_interaction": "Candidate-specific prior count or prior-seen interaction features.",
    "brand_prior": "Candidate-brand affinity features derived from strictly prior user history; candidate brand presence is excluded.",
}
semantic_group_definition_df = pd.DataFrame([
    {
        "semantic_group": semantic_group,
        "definition": semantic_group_definitions[semantic_group],
        "available": bool(semantic_group_to_features[semantic_group]),
        "feature_count": int(len(semantic_group_to_features[semantic_group])),
        "features_json": json.dumps(semantic_group_to_features[semantic_group], ensure_ascii=False),
        "unavailable_reason": (
            "No direct query-profile interaction feature exists in the executed model contract; no proxy was invented."
            if semantic_group == "query_profile_interaction" and not semantic_group_to_features[semantic_group]
            else ""
        ),
        "definition_selected_without_outcomes": True,
    }
    for semantic_group in REQUIRED_SEMANTIC_GROUPS
])

facet_family_names = sorted({
    feature.split("__", 1)[1]
    for feature in feature_names
    if feature.startswith(("qmatch__", "uaff__", "ucountlog__")) and "__" in feature
})
if not facet_family_names or "brand" in facet_family_names:
    raise RuntimeError("Functional facet families are empty or incorrectly include brand.")
facet_family_to_features = {}
for family in facet_family_names:
    allowed = {
        f"qmatch__{family}", f"uaff__{family}", f"ucountlog__{family}",
        f"user_{family}_recency_days", f"user_{family}_recent_count_180d",
    }
    facet_family_to_features[family] = [feature for feature in feature_names if feature in allowed]
    if not facet_family_to_features[family]:
        raise RuntimeError(f"Facet family has no executed features: {family}")
facet_family_mapping_df = pd.DataFrame([
    {
        "facet_family": family,
        "feature": feature,
        "feature_role": facet_feature_role(feature),
        "brand_excluded": True,
        "definition_selected_without_outcomes": True,
    }
    for family, features in facet_family_to_features.items()
    for feature in features
])

model_rows = interpretation_manifest.get("fold_models", [])
model_key_rows = [(int(row["pool_depth"]), int(row["fold_id"])) for row in model_rows]
if not model_key_rows or len(model_key_rows) != len(set(model_key_rows)):
    raise RuntimeError("Fold-model artifact keys are empty or duplicated.")
model_artifact_map = {
    (int(row["pool_depth"]), int(row["fold_id"])): Path(row["path"])
    for row in model_rows
}
missing_models = [str(path) for path in model_artifact_map.values() if not path.exists()]
if missing_models:
    raise FileNotFoundError("Missing fold models:\n" + "\n".join(missing_models))

canonical_raw_df = pd.read_parquet(CANONICAL_RAW_PATH)
require_columns(
    canonical_raw_df,
    ["case_id", "regime", "stage_condition", "method_family"],
    "Notebook 14 canonical raw",
)
canonical_condition_df = canonical_raw_df.loc[
    canonical_raw_df["stage_condition"].astype(str).eq(condition_name)
    & canonical_raw_df["method_family"].astype(str).eq(MODEL_FAMILY)
].copy()
if canonical_condition_df.empty:
    raise RuntimeError("Notebook 14 contains no RankP LightGBM rows for regime lineage.")
if canonical_condition_df.groupby("case_id")["regime"].nunique(dropna=False).gt(1).any():
    raise RuntimeError("Notebook 14 regime labels vary within case_id.")
regime_map_df = canonical_condition_df[["case_id", "regime"]].drop_duplicates("case_id").copy()
regime_map_df["case_id"] = regime_map_df["case_id"].astype(str)

user_id_grain_available = "user_id" in canonical_condition_df.columns
if user_id_grain_available:
    case_user_df = canonical_condition_df[["case_id", "user_id"]].drop_duplicates().copy()
    if case_user_df.duplicated("case_id").any() or case_user_df.duplicated("user_id").any():
        raise RuntimeError("Notebook 14 violates the one-case-per-user evaluation grain.")

artifact_specs = interpretation_manifest.get("candidate_feature_oof_artifacts", [])
artifact_depths = [int(artifact["pool_depth"]) for artifact in artifact_specs]
if not artifact_specs or len(artifact_depths) != len(set(artifact_depths)):
    raise RuntimeError("Held-out candidate artifacts are empty or duplicated by pool depth.")
if PRIMARY_POOL_DEPTH not in artifact_depths:
    raise RuntimeError(f"Primary pool depth {PRIMARY_POOL_DEPTH} is missing from the interpretation manifest.")

heldout_frames = []
expected_case_universe = None
for artifact in artifact_specs:
    artifact_path = Path(artifact["path"])
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing held-out candidate artifact: {artifact_path}")
    frame = pd.read_parquet(artifact_path)
    required = [
        "condition_name", "category_id", "pool_depth", "case_id", "query_id",
        "candidate_parent_asin", "label", "fold_id", "oof_prediction",
        "model_oof_prediction", "prediction_source", *feature_names,
    ]
    require_columns(frame, required, str(artifact_path))
    require_unique_columns(frame, str(artifact_path))
    frame["pool_depth"] = pd.to_numeric(frame["pool_depth"], errors="raise").astype(int)
    frame["fold_id"] = pd.to_numeric(frame["fold_id"], errors="raise").astype(int)
    frame["case_id"] = frame["case_id"].astype(str)
    frame["query_id"] = frame["query_id"].astype(str)
    frame["candidate_parent_asin"] = frame["candidate_parent_asin"].astype(str)
    frame["prediction_source"] = frame["prediction_source"].fillna("").astype(str)
    if set(frame["pool_depth"]) != {int(artifact["pool_depth"])}:
        raise RuntimeError(f"Pool-depth lineage mismatch in {artifact_path}.")
    if int(artifact.get("row_count", len(frame))) != int(len(frame)):
        raise RuntimeError(f"Manifest row count differs from {artifact_path}.")
    if int(artifact.get("case_count", frame["case_id"].nunique())) != int(frame["case_id"].nunique()):
        raise RuntimeError(f"Manifest case count differs from {artifact_path}.")
    case_universe = set(frame["case_id"])
    if expected_case_universe is None:
        expected_case_universe = case_universe
    elif case_universe != expected_case_universe:
        raise RuntimeError("Held-out candidate artifacts do not preserve one common case universe.")
    row_count_before_join = len(frame)
    frame = frame.merge(regime_map_df, on="case_id", how="left", validate="many_to_one")
    if len(frame) != row_count_before_join:
        raise RuntimeError(f"Notebook 14 regime join multiplied rows for {artifact_path}.")
    if frame["regime"].isna().any():
        raise RuntimeError(f"Notebook 14 regime join failed for {artifact_path}.")
    unique_key = ["pool_depth", "case_id", "candidate_parent_asin"]
    if frame.duplicated(unique_key).any():
        raise RuntimeError(f"Held-out candidate rows are duplicated in {artifact_path}.")
    heldout_frames.append(frame)

heldout_df = pd.concat(heldout_frames, ignore_index=True, sort=False)
if set(heldout_df["category_id"].astype(str)) != {CATEGORY_ID}:
    raise RuntimeError("Held-out artifact category mismatch.")
if set(heldout_df["condition_name"].astype(str)) != {condition_name}:
    raise RuntimeError("Held-out artifact condition mismatch.")
if heldout_df[["case_id", "query_id", "candidate_parent_asin"]].eq("").any().any():
    raise RuntimeError("Held-out identifiers must be non-empty.")

case_query_df = heldout_df[["case_id", "query_id"]].drop_duplicates()
if case_query_df.duplicated("case_id").any() or case_query_df.duplicated("query_id").any():
    raise RuntimeError("case_id and query_id must remain one-to-one.")
if heldout_df.groupby(["pool_depth", "case_id"])["fold_id"].nunique().gt(1).any():
    raise RuntimeError("A case is assigned to more than one fold within a pool depth.")
if heldout_df.groupby(["pool_depth", "case_id"])["prediction_source"].nunique().gt(1).any():
    raise RuntimeError("prediction_source varies within a case and pool depth.")
candidate_counts = heldout_df.groupby(["pool_depth", "case_id"]).size()
if any(int(count) > int(depth) for (depth, _), count in candidate_counts.items()):
    raise RuntimeError("A case contains more candidate rows than its declared pool depth.")

observed_prediction_sources = set(heldout_df["prediction_source"])
expected_prediction_sources = {NATIVE_PREDICTION_SOURCE, FALLBACK_PREDICTION_SOURCE}
LEGACY_LIGHTGBM_OOF_SOURCE = "lightgbm_oof"
legacy_mask = heldout_df["prediction_source"].eq(LEGACY_LIGHTGBM_OOF_SOURCE)
if legacy_mask.any():
    heldout_df.loc[
        legacy_mask & heldout_df["regime"].ne("cold"),
        "prediction_source",
    ] = NATIVE_PREDICTION_SOURCE
    heldout_df.loc[
        legacy_mask & heldout_df["regime"].eq("cold"),
        "prediction_source",
    ] = FALLBACK_PREDICTION_SOURCE

observed_prediction_sources = set(heldout_df["prediction_source"])
expected_prediction_sources = {NATIVE_PREDICTION_SOURCE, FALLBACK_PREDICTION_SOURCE}
if not observed_prediction_sources.issubset(expected_prediction_sources):
    raise RuntimeError(
        f"Unexpected P2-P prediction sources: {sorted(observed_prediction_sources - expected_prediction_sources)}"
    )
native_heldout_df = heldout_df.loc[
    heldout_df["prediction_source"].eq(NATIVE_PREDICTION_SOURCE)
].copy()
fallback_heldout_df = heldout_df.loc[
    heldout_df["prediction_source"].eq(FALLBACK_PREDICTION_SOURCE)
].copy()
if native_heldout_df.empty:
    raise RuntimeError("No native RankP LightGBM rows are available for held-out interpretation.")
primary_native_df = native_heldout_df.loc[native_heldout_df["pool_depth"].eq(PRIMARY_POOL_DEPTH)]
if primary_native_df.empty:
    raise RuntimeError(f"No native rows are available at primary pool depth {PRIMARY_POOL_DEPTH}.")
if "strong" not in set(primary_native_df["regime"].astype(str)):
    raise RuntimeError("Strong-regime native rows are required at the primary report depth.")

fold_assignment_path = Path(interpretation_manifest["fold_assignment_path"])
if not fold_assignment_path.exists():
    raise FileNotFoundError(f"Missing fold-assignment artifact: {fold_assignment_path}")
fold_assignment_df = pd.read_parquet(fold_assignment_path)
require_columns(
    fold_assignment_df,
    ["condition_name", "category_id", "pool_depth", "case_id", "query_id", "fold_id"],
    "fold-assignment artifact",
)
fold_assignment_df["pool_depth"] = pd.to_numeric(fold_assignment_df["pool_depth"], errors="raise").astype(int)
fold_assignment_df["fold_id"] = pd.to_numeric(fold_assignment_df["fold_id"], errors="raise").astype(int)
fold_assignment_df["case_id"] = fold_assignment_df["case_id"].astype(str)
fold_assignment_df["query_id"] = fold_assignment_df["query_id"].astype(str)
if fold_assignment_df.duplicated(["pool_depth", "case_id"]).any():
    raise RuntimeError("Fold-assignment artifact duplicates a case within a pool depth.")
observed_lineage = heldout_df[
    ["condition_name", "category_id", "pool_depth", "case_id", "query_id", "fold_id"]
].drop_duplicates()
lineage_columns = ["condition_name", "category_id", "pool_depth", "case_id", "query_id", "fold_id"]
observed_lineage_set = set(map(tuple, observed_lineage[lineage_columns].astype(str).to_numpy()))
declared_lineage_set = set(map(tuple, fold_assignment_df[lineage_columns].astype(str).to_numpy()))
if observed_lineage_set != declared_lineage_set:
    raise RuntimeError("Held-out candidate fold lineage differs from the declared fold assignments.")

observed_model_keys = set(
    map(tuple, heldout_df[["pool_depth", "fold_id"]].astype(int).drop_duplicates().to_numpy())
)
if not observed_model_keys.issubset(model_artifact_map):
    raise RuntimeError(
        f"Held-out folds lack fold-model artifacts: {sorted(observed_model_keys - set(model_artifact_map))}"
    )


In [6]:
# ==== Load Fold Boosters and Extract Gain Importance ====
gain_rows = []
model_cache = {}
for (pool_depth, fold_id), model_path in sorted(model_artifact_map.items()):
    booster = lgb.Booster(model_file=str(model_path))
    if booster.feature_name() != feature_names:
        raise RuntimeError(f"Fold model feature order mismatch: {model_path}")
    model_cache[(pool_depth, fold_id)] = booster
    gain = booster.feature_importance(importance_type="gain").astype(float)
    gain_normalized = gain / gain.sum() if gain.sum() > 0 else np.zeros_like(gain)
    rank = pd.Series(-gain_normalized).rank(method="average").to_numpy()
    for position, feature in enumerate(feature_names):
        gain_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "model_family": MODEL_FAMILY,
            "candidate_pool_depth": int(pool_depth),
            "fold_id": int(fold_id),
            "feature": feature,
            "feature_group": feature_to_group[feature],
            "gain": float(gain[position]),
            "normalized_gain": float(gain_normalized[position]),
            "gain_rank": float(rank[position]),
            "importance_scope": "fold_model_intrinsic_training_derived_not_a_heldout_metric",
            "evidence_role": "supporting_training_derived",
            "causal_claim": False,
        })
gain_by_fold_df = pd.DataFrame(gain_rows)

correlation_rows = []
stability_rows = []
for pool_depth, depth_df in gain_by_fold_df.groupby("candidate_pool_depth", observed=True):
    rank_pivot = depth_df.pivot(index="feature", columns="fold_id", values="gain_rank")
    correlation = rank_pivot.corr(method="spearman")
    for left_fold in correlation.index:
        for right_fold in correlation.columns:
            if int(left_fold) < int(right_fold):
                correlation_rows.append({
                    "category": CATEGORY_LABEL,
                    "category_id": CATEGORY_ID,
                    "candidate_pool_depth": int(pool_depth),
                    "left_fold_id": int(left_fold),
                    "right_fold_id": int(right_fold),
                    "spearman_rank_correlation": float(correlation.loc[left_fold, right_fold]),
                })
    for (feature, group), subset in depth_df.groupby(["feature", "feature_group"], observed=True):
        mean_gain = float(subset["normalized_gain"].mean())
        std_gain = float(subset["normalized_gain"].std(ddof=0))
        stability_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "model_family": MODEL_FAMILY,
            "candidate_pool_depth": int(pool_depth),
            "feature": feature,
            "feature_group": group,
            "fold_count": int(subset["fold_id"].nunique()),
            "mean_normalized_gain": mean_gain,
            "std_normalized_gain": std_gain,
            "coefficient_of_variation": float(std_gain / mean_gain) if mean_gain > 0 else np.nan,
            "mean_gain_rank": float(subset["gain_rank"].mean()),
        })
fold_rank_correlation_df = pd.DataFrame(correlation_rows)
gain_stability_df = pd.DataFrame(stability_rows)


In [7]:
# ==== Native fold-model Reproduction / leakage-integrity Gate (kept). Permutation Importance (slow held-out re-scoring) removed. ====
reproduction_rows = []
for (pool_depth, fold_id), fold_df in native_heldout_df.groupby(
    ["pool_depth", "fold_id"], observed=True
):
    pool_depth = int(pool_depth)
    fold_id = int(fold_id)
    booster = model_cache[(pool_depth, fold_id)]
    fold_df = fold_df.reset_index(drop=True)
    x_heldout = fold_df[feature_names].astype(np.float32)
    baseline_prediction = booster.predict(x_heldout, num_iteration=booster.best_iteration or -1)
    expected_model_prediction = pd.to_numeric(
        fold_df["model_oof_prediction"], errors="raise"
    ).to_numpy(dtype=float)
    expected_evaluated_prediction = pd.to_numeric(
        fold_df["oof_prediction"], errors="raise"
    ).to_numpy(dtype=float)
    maximum_model_error = float(np.max(np.abs(baseline_prediction - expected_model_prediction)))
    maximum_evaluated_error = float(np.max(np.abs(baseline_prediction - expected_evaluated_prediction)))
    maximum_native_export_error = float(
        np.max(np.abs(expected_model_prediction - expected_evaluated_prediction))
    )
    reproduction_passed = bool(
        maximum_model_error <= REPRODUCTION_ATOL
        and maximum_evaluated_error <= REPRODUCTION_ATOL
        and maximum_native_export_error <= REPRODUCTION_ATOL
    )
    fallback_row_count = int(len(fallback_heldout_df.loc[
        fallback_heldout_df["pool_depth"].eq(pool_depth)
        & fallback_heldout_df["fold_id"].eq(fold_id)
    ]))
    reproduction_rows.append({
        "candidate_pool_depth": pool_depth,
        "fold_id": fold_id,
        "native_heldout_row_count": int(len(fold_df)),
        "fallback_row_count_excluded": fallback_row_count,
        "native_prediction_source": NATIVE_PREDICTION_SOURCE,
        "excluded_prediction_source": FALLBACK_PREDICTION_SOURCE,
        "maximum_absolute_model_oof_error": maximum_model_error,
        "maximum_absolute_evaluated_oof_error": maximum_evaluated_error,
        "maximum_absolute_native_export_error": maximum_native_export_error,
        "reproduction_tolerance": REPRODUCTION_ATOL,
        "prediction_reproduced": reproduction_passed,
    })
    if not reproduction_passed:
        raise RuntimeError(
            "Native fold-model predictions do not reproduce the strict OOF exports at "
            f"depth={pool_depth}, fold={fold_id}: "
            f"model_error={maximum_model_error}, evaluated_error={maximum_evaluated_error}, "
            f"export_error={maximum_native_export_error}"
        )

reproduction_qc_df = pd.DataFrame(reproduction_rows)


In [8]:
# ==== Gain-only Assembly and Export (permutation + SHAP removed; slim) Fig 7.1 Source = Grouped normalized-gain Shares (training-derived, Deterministic from the Fold Boosters -> bit-identical to the pre-slim Gain path). No held-out re-scoring Remains in This notebook. ====
gain_group_df = (
    gain_by_fold_df.groupby(
        ["category", "category_id", "condition_name", "model_family",
         "candidate_pool_depth", "fold_id", "feature_group"],
        as_index=False,
        observed=True,
    )["normalized_gain"].sum()
    .rename(columns={"normalized_gain": "normalized_gain_group_sum"})
)

# QC: grouped gain shares sum to 1 within each (depth, fold), or 0 if a fold
# booster reports zero total gain.
_group_share_totals = (
    gain_group_df.groupby(["candidate_pool_depth", "fold_id"], observed=True)[
        "normalized_gain_group_sum"
    ]
    .sum()
    .to_numpy(dtype=float)
)
if not bool(
    np.all(
        np.isclose(_group_share_totals, 1.0, atol=1e-6)
        | np.isclose(_group_share_totals, 0.0, atol=1e-6)
    )
):
    raise RuntimeError("Grouped gain shares do not sum to one within a depth/fold.")

feature_group_gain_summary_df = (
    gain_group_df.groupby(
        ["category", "category_id", "condition_name", "model_family",
         "candidate_pool_depth", "feature_group"],
        as_index=False,
        observed=True,
    ).agg(
        fold_count=("fold_id", "nunique"),
        mean_normalized_gain=("normalized_gain_group_sum", "mean"),
        std_normalized_gain_across_folds=(
            "normalized_gain_group_sum",
            lambda values: float(np.std(values, ddof=0)),
        ),
        min_normalized_gain_fold=("normalized_gain_group_sum", "min"),
        max_normalized_gain_fold=("normalized_gain_group_sum", "max"),
    )
)
feature_group_gain_summary_df["primary_report_depth"] = (
    feature_group_gain_summary_df["candidate_pool_depth"].eq(PRIMARY_POOL_DEPTH)
)
feature_group_gain_summary_df["gain_evidence_role"] = "supporting_training_derived"
feature_group_gain_summary_df["importance_scope"] = (
    "fold_model_intrinsic_training_derived_not_a_heldout_metric"
)
feature_group_gain_summary_df["causal_claim"] = False

primary_feature_group_gain_summary_df = feature_group_gain_summary_df.loc[
    feature_group_gain_summary_df["candidate_pool_depth"].eq(PRIMARY_POOL_DEPTH)
].copy()
if primary_feature_group_gain_summary_df.empty:
    raise RuntimeError("No grouped gain summary at the primary report depth.")
_primary_share_total = float(
    primary_feature_group_gain_summary_df["mean_normalized_gain"].sum()
)
if not (
    np.isclose(_primary_share_total, 1.0, atol=1e-6)
    or np.isclose(_primary_share_total, 0.0, atol=1e-6)
):
    raise RuntimeError(
        f"Primary grouped gain shares do not sum to one: {_primary_share_total}"
    )

qc_df = pd.concat([
    reproduction_qc_df.assign(check="native_fold_prediction_reproduction", severity="error"),
    pd.DataFrame([
        {
            "check": "cold_fallback_excluded_from_native_interpretation",
            "severity": "error",
            "prediction_reproduced": True,
            "native_heldout_row_count": int(len(native_heldout_df)),
            "fallback_row_count_excluded": int(len(fallback_heldout_df)),
        },
        {
            "check": "brand_prior_reported_separately",
            "severity": "error",
            "prediction_reproduced": bool(semantic_group_to_features["brand_prior"]),
            "native_heldout_row_count": int(len(native_heldout_df)),
        },
        {
            "check": "primary_pool_depth_1000_reported",
            "severity": "error",
            "prediction_reproduced": bool(
                primary_feature_group_gain_summary_df["candidate_pool_depth"].eq(PRIMARY_POOL_DEPTH).all()
            ),
            "native_heldout_row_count": int(len(primary_native_df)),
        },
        {
            "check": "grouped_gain_shares_sum_to_one",
            "severity": "error",
            "prediction_reproduced": True,
            "native_heldout_row_count": int(len(primary_native_df)),
        },
        {
            "check": "query_profile_interaction_availability",
            "severity": "information",
            "prediction_reproduced": bool(semantic_group_to_features["query_profile_interaction"]),
            "native_heldout_row_count": int(len(primary_native_df)),
        },
        {
            "check": "case_query_one_to_one",
            "severity": "error",
            "prediction_reproduced": True,
            "native_heldout_row_count": int(len(native_heldout_df)),
        },
        {
            "check": "case_user_one_to_one_available_in_notebook14",
            "severity": "information",
            "prediction_reproduced": bool(user_id_grain_available),
            "native_heldout_row_count": int(len(native_heldout_df)),
        },
    ]),
], ignore_index=True, sort=False)

output_frames = {
    "gain_by_fold": gain_by_fold_df,
    "gain_stability": gain_stability_df,
    "fold_rank_correlation": fold_rank_correlation_df,
    "feature_group_gain_summary": feature_group_gain_summary_df,
    "primary_feature_group_gain_summary": primary_feature_group_gain_summary_df,
    "qc": qc_df,
}
for name, frame in output_frames.items():
    require_unique_columns(frame, name)
    frame.to_csv(OUTPUT_FILES[name], index=False, encoding="utf-8-sig")

run_manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category": CATEGORY_LABEL,
    "model_family": MODEL_FAMILY,
    "condition_name": condition_name,
    "source_interpretation_manifest": str(INTERPRETATION_MANIFEST_PATH),
    "source_notebook14_canonical_raw": str(CANONICAL_RAW_PATH),
    "primary_metric": "NDCG@5",
    "primary_report_pool_depth": PRIMARY_POOL_DEPTH,
    "native_prediction_source": NATIVE_PREDICTION_SOURCE,
    "excluded_prediction_source": FALLBACK_PREDICTION_SOURCE,
    "native_heldout_row_count": int(len(native_heldout_df)),
    "fallback_row_count_excluded": int(len(fallback_heldout_df)),
    "fold_prediction_reproduction_tolerance": REPRODUCTION_ATOL,
    "importance_method": "fold_model_intrinsic_gain",
    "interpretation_scope": "training_derived_gain_only",
    "permutation_importance_computed": False,
    "shap_computed": False,
    "feature_groups": LEGACY_INTERPRETATION_GROUPS,
    "brand_prior_reported_separately": True,
    "gain_evidence_role": "supporting_training_derived",
    "causal_claim": False,
    "grouped_gain_shares_sum_to_one": True,
    "output_files": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
OUTPUT_FILES["manifest"].write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Held-out LightGBM interpretation (gain-only slim) written to:", OUT_DIR)


Held-out LightGBM interpretation (gain-only slim) written to: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/lightgbm_interpretation


In [9]:
# ==== Batch C2 — Interpretation Guard (Exact-item Familiarity Exclusion) ====
# Batch C2 interpretation guard
_exact_item_in_interpretation = sorted(
    set(feature_names).intersection(EXACT_ITEM_FAMILIARITY_FEATURES)
)
if _exact_item_in_interpretation:
    raise RuntimeError(
        f"LightGBM interpretation contains previously-reviewed-item diagnostics: {_exact_item_in_interpretation}"
    )
print("LightGBM primary interpretation seen-item exclusion QC passed.")


LightGBM primary interpretation seen-item exclusion QC passed.


In [10]:
# ==== Final Verification Report Export ====
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math

try:
    import numpy as _report_np
except Exception:
    _report_np = None
try:
    import pandas as _report_pd
except Exception:
    _report_pd = globals().get("pd")


def _report_is_dataframe(value):
    return _report_pd is not None and isinstance(value, _report_pd.DataFrame)


def _report_is_missing(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if _report_pd is not None:
        try:
            missing = _report_pd.isna(value)
            if isinstance(missing, (bool, type(None))):
                return bool(missing)
        except Exception:
            pass
    return False


def _report_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if _report_np is not None and isinstance(value, _report_np.generic):
        return _report_jsonable(value.item())
    if _report_is_missing(value):
        return None
    if isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return None if math.isnan(value) else value
    if isinstance(value, dict):
        return {str(k): _report_jsonable(v) for k, v in value.items()}
    if _report_pd is not None:
        if isinstance(value, _report_pd.Series):
            return [_report_jsonable(v) for v in value.tolist()]
        if _report_is_dataframe(value):
            return _report_frame_records(value)
    if isinstance(value, (list, tuple, set)):
        return [_report_jsonable(v) for v in value]
    if hasattr(value, "tolist"):
        try:
            return _report_jsonable(value.tolist())
        except Exception:
            pass
    if hasattr(value, "item"):
        try:
            return _report_jsonable(value.item())
        except Exception:
            pass
    return str(value)


def _report_frame_records(frame, limit=50, columns=None, drop_case_columns=True):
    if not _report_is_dataframe(frame):
        return []
    work = frame.copy()
    if columns is not None:
        keep = [column for column in columns if column in work.columns]
        work = work[keep]
    if drop_case_columns:
        disallowed = {
            "case_id", "query_id", "user_id", "item_id", "parent_asin",
            "target_parent_asin", "target_item_id", "review_id",
        }
        drop = [column for column in work.columns if str(column).lower() in disallowed]
        if drop:
            work = work.drop(columns=drop)
    records = [_report_jsonable(row) for row in work.head(limit).to_dict(orient="records")]
    if len(work) > limit:
        records.append({"note": "truncated", "row_count": int(len(work)), "rows_emitted": int(limit)})
    return records


def _report_output_dir():
    if "OUT_DIR" in globals():
        return Path(globals()["OUT_DIR"])
    if "OUTPUT_DIR" in globals():
        return Path(globals()["OUTPUT_DIR"])
    category_key = globals().get("CATEGORY_KEY", "facial_skincare")
    project_root = Path(globals().get(
        "PROJECT_ROOT",
        f"/content/drive/MyDrive/thesis_recsys/categories/{category_key}",
    ))
    out_dir = project_root / "outputs" / "analysis" / "lightgbm_interpretation"
    out_dir.mkdir(parents=True, exist_ok=True)
    globals()["PROJECT_ROOT"] = project_root
    globals()["OUT_DIR"] = out_dir
    return out_dir


_report_dir = _report_output_dir() / "report"
_report_dir.mkdir(parents=True, exist_ok=True)


def _report_is_inside(path, parent):
    try:
        Path(path).resolve().relative_to(Path(parent).resolve())
        return True
    except Exception:
        return False


def _report_file_sha256(path, allow_heavy=False):
    try:
        path = Path(path)
    except Exception:
        return None
    if not path.exists() or not path.is_file():
        return None
    if not allow_heavy and path.suffix.lower() in {".parquet", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}:
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _report_notebook_path():
    candidates = []
    for key in ["__vsc_ipynb_file__", "__file__"]:
        value = globals().get(key)
        if value:
            candidates.append(Path(value))
    notebook_name = globals().get("NOTEBOOK_NAME")
    if notebook_name:
        candidates.append(Path.cwd() / str(notebook_name))
        project_root = globals().get("PROJECT_ROOT")
        if project_root:
            candidates.append(Path(project_root) / str(notebook_name))
    for candidate in candidates:
        try:
            if candidate.exists() and candidate.suffix.lower() == ".ipynb":
                return candidate
        except Exception:
            pass
    return None


_report_nb_path = _report_notebook_path()
_report_notebook = _report_nb_path.name if _report_nb_path is not None else str(globals().get("NOTEBOOK_NAME", "unknown_notebook"))
_report_category = globals().get("CATEGORY_ID", globals().get("CATEGORY_LABEL", "cross_category"))


def _report_path_from_maps(key):
    for map_name in ["OUTPUT_FILES", "OUTPUT_PATHS", "output_paths"]:
        mapping = globals().get(map_name)
        if isinstance(mapping, dict) and key in mapping:
            return str(mapping[key])
    return None


def _report_path_from_var(name):
    value = globals().get(name)
    return str(value) if value is not None else None


def _report_row_value(row, candidates):
    for column in candidates:
        if column in row and not _report_is_missing(row[column]):
            return row[column]
    return None


def _report_ci(row, low_candidates, high_candidates):
    lo = _report_row_value(row, low_candidates)
    hi = _report_row_value(row, high_candidates)
    if _report_is_missing(lo) or _report_is_missing(hi):
        return None
    return [_report_jsonable(lo), _report_jsonable(hi)]


def _report_value(claim_id, value=None, ci=None, p=None, n=None, source_file=None, aggregation=None, note=None):
    record = {
        "claim_id": str(claim_id),
        "value": _report_jsonable(value),
        "ci": _report_jsonable(ci),
        "p": _report_jsonable(p),
        "n": _report_jsonable(n),
        "source_file": _report_jsonable(source_file),
        "aggregation": _report_jsonable(aggregation),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_missing_value(claim_id, note="not_available_in_notebook"):
    return _report_value(
        claim_id=claim_id,
        value=None,
        ci=None,
        p=None,
        n=None,
        source_file=None,
        aggregation="not_available_in_notebook",
        note=note,
    )


def _report_gate(gate_id, observed=None, expected_contract="", self_flag=False, note=None):
    record = {
        "gate_id": str(gate_id),
        "observed": _report_jsonable(observed),
        "expected_contract": str(expected_contract),
        "self_flag": bool(self_flag),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_frame_failed(frame, passed_columns=("passed", "check_passed", "identity_passed"), status_columns=("status", "coverage_status")):
    if not _report_is_dataframe(frame) or frame.empty:
        return False
    for column in passed_columns:
        if column in frame.columns:
            try:
                return not bool(frame[column].astype(bool).all())
            except Exception:
                pass
    for column in status_columns:
        if column in frame.columns:
            statuses = frame[column].astype(str).str.upper()
            return not bool(statuses.isin(["PASS", "SUCCESS", "TRUE"]).all())
    return False


def _report_filter_primary(frame):
    work = frame.copy()
    original = work
    if "metric_name" in work.columns:
        filtered = work.loc[work["metric_name"].astype(str).eq(str(globals().get("PRIMARY_METRIC_NAME", "NDCG")))]
        if not filtered.empty:
            work = filtered
    if "metric_cutoff" in work.columns:
        filtered = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(int(globals().get("PRIMARY_METRIC_CUTOFF", 5)))]
        if not filtered.empty:
            work = filtered
    if "candidate_pool_depth" in work.columns and "REPORT_POOL_DEPTH" in globals():
        filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["REPORT_POOL_DEPTH"]))]
        if not filtered.empty:
            work = filtered
    return work if not work.empty else original


def _report_values_15():
    values = []
    five = globals().get("stage_allocation_five_condition_comparison_df")
    if _report_is_dataframe(five):
        work = _report_filter_primary(five)
        value_columns = ["mean_metric_value", "metric_mean", "mean_value", "mean", "metric_value_mean", "mean_ndcg_at_5"]
        value_column = next((column for column in value_columns if column in work.columns), None)
        if value_column is None:
            values.append(_report_missing_value("section6_five_condition_means", "value_column_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [
                    row.get("stage_condition"), row.get("reranker_family"),
                    row.get("candidate_pool_depth"), row.get("metric_name"), row.get("metric_cutoff"),
                ]
                claim_id = "section6_condition_mean:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get(value_column),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low", "lower_ci"], ["bootstrap_ci_95_high", "ci_high", "upper_ci"]),
                    p=_report_row_value(row, ["p", "p_value", "sign_flip_p_value"]),
                    n=_report_row_value(row, ["case_count", "query_count", "n_cases", "n", "sample_size"]),
                    source_file=_report_path_from_maps("five_condition_comparison"),
                    aggregation=f"{value_column} from stage_allocation_five_condition_comparison_df",
                ))
    else:
        values.append(_report_missing_value("section6_five_condition_means"))

    contrasts = globals().get("stage_allocation_paired_contrasts_df")
    if _report_is_dataframe(contrasts):
        work = contrasts.copy()
        if "contrast_name" in work.columns:
            work = work.loc[work["contrast_name"].astype(str).eq("P2-P_minus_P2-Q")]
        if "metric_name" in work.columns:
            work = work.loc[work["metric_name"].astype(str).eq("NDCG")]
        if "metric_cutoff" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(5)]
        if "candidate_pool_depth" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").isin([100, 300, 500, 700, 1000])]
        if "user_scope" in work.columns:
            work = work.loc[work["user_scope"].astype(str).eq("overall")]
        if "analysis_subset" in work.columns:
            work = work.loc[work["analysis_subset"].astype(str).eq("all_cases")]
        if work.empty:
            values.append(_report_missing_value("fig6_3_p2p_minus_p2q_ndcg5_by_depth", "requested_contrast_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                claim_id = "fig6_3_p2p_minus_p2q_ndcg5_by_depth:" + "|".join(str(row.get(column)) for column in ["reranker_family", "candidate_pool_depth"] if column in row)
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("mean_difference"),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low"], ["bootstrap_ci_95_high", "ci_high"]),
                    p=_report_row_value(row, ["sign_flip_p_value", "p_value", "p"]),
                    n=_report_row_value(row, ["paired_query_count", "n_pairs", "case_count", "n"]),
                    source_file=_report_path_from_maps("paired_contrasts"),
                    aggregation="existing paired mean_difference for P2-P_minus_P2-Q by depth",
                ))
    else:
        values.append(_report_missing_value("fig6_3_p2p_minus_p2q_ndcg5_by_depth"))
    return values


def _report_gates_15():
    gates = []
    manifest_obj = globals().get("manifest", {}) if isinstance(globals().get("manifest", {}), dict) else {}
    sig = globals().get("stage_allocation_significance_tests_df")
    observed_scope = {
        "analysis_role": manifest_obj.get("analysis_role"),
        "confirmatory_inference_authority_values": sorted(sig["confirmatory_inference_authority"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "confirmatory_inference_authority" in sig.columns else None,
        "inference_role_values": sorted(sig["inference_role"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "inference_role" in sig.columns else None,
    }
    gates.append(_report_gate("descriptive_only_scope", observed_scope, "descriptive-only; confirmatory inference authority remains Notebook 16", False))
    authority = globals().get("aggregate_metric_authority_qc_df")
    gates.append(_report_gate(
        "aggregate_metric_authority_qc",
        _report_frame_records(authority, limit=30),
        "descriptive aggregates reconcile with canonical authority",
        _report_frame_failed(authority, passed_columns=("check_passed", "passed")),
        None if _report_is_dataframe(authority) else "not_available_in_notebook",
    ))
    identity = globals().get("cold_fallback_identity_qc_df")
    gates.append(_report_gate(
        "cold_fallback_identity_qc",
        _report_frame_records(identity, limit=30),
        "fallback identity diagnostics are descriptive QC only",
        _report_frame_failed(identity),
        None if _report_is_dataframe(identity) else "not_available_in_notebook",
    ))
    gates.append(_report_gate(
        "stage_delta_fold_lineage_qc",
        manifest_obj.get("stage_delta_fold_lineage_qc", globals().get("stage_delta_fold_lineage_qc")),
        "stage-delta lineage recorded from existing Notebook 14/15 artifacts",
        False,
        None if (manifest_obj.get("stage_delta_fold_lineage_qc") is not None or "stage_delta_fold_lineage_qc" in globals()) else "not_available_in_notebook",
    ))
    return gates


def _report_values_18():
    values = []
    frame = globals().get("primary_feature_group_gain_summary_df")
    source_key = "primary_feature_group_gain_summary"
    if not _report_is_dataframe(frame):
        frame = globals().get("feature_group_gain_summary_df")
        source_key = "feature_group_gain_summary"
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "primary_report_depth" in work.columns:
            filtered = work.loc[work["primary_report_depth"].astype(bool)]
            if not filtered.empty:
                work = filtered
        elif "candidate_pool_depth" in work.columns and "PRIMARY_POOL_DEPTH" in globals():
            filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["PRIMARY_POOL_DEPTH"]))]
            if not filtered.empty:
                work = filtered
        if "feature_group" not in work.columns or "mean_normalized_gain" not in work.columns:
            values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct", "required_columns_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"fig7_1_grouped_normalized_gain_pct:{row.get('feature_group')}",
                    value=None if _report_is_missing(row.get("mean_normalized_gain")) else float(row.get("mean_normalized_gain")) * 100.0,
                    ci=None,
                    p=None,
                    n=_report_row_value(row, ["fold_count", "n_folds", "n"]),
                    source_file=_report_path_from_maps(source_key),
                    aggregation="mean_normalized_gain across folds, expressed as percent",
                ))
    else:
        values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct"))
    return values


def _report_gates_18():
    gates = []
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    share_observed = {
        "manifest_flag": run.get("grouped_gain_shares_sum_to_one"),
        "primary_share_total": globals().get("_primary_share_total"),
        "group_share_totals": globals().get("_group_share_totals"),
    }
    share_self_flag = False
    if share_observed["manifest_flag"] is not None:
        share_self_flag = share_observed["manifest_flag"] is not True
    gates.append(_report_gate("grouped_gain_shares_sum_to_one", share_observed, "grouped gain shares sum to one within existing fold/depth summaries", share_self_flag))
    perm_flag = run.get("permutation_importance_computed")
    gates.append(_report_gate("permutation_importance_computed", perm_flag, "false in slim build manifest", perm_flag is not False, None if perm_flag is not None else "not_available_in_notebook"))
    shap_flag = run.get("shap_computed")
    gates.append(_report_gate("shap_computed", shap_flag, "false in slim build manifest", shap_flag is not False, None if shap_flag is not None else "not_available_in_notebook"))
    reproduction = globals().get("reproduction_qc_df")
    gates.append(_report_gate(
        "native_fold_prediction_reproduction",
        _report_frame_records(reproduction, limit=30),
        "native fold predictions reproduce strict OOF exports within recorded tolerance",
        _report_frame_failed(reproduction, passed_columns=("prediction_reproduced", "passed")),
        None if _report_is_dataframe(reproduction) else "not_available_in_notebook",
    ))
    return gates


def _report_values_19():
    values = []
    frame = globals().get("branch_contribution_df")
    required = list(globals().get("REQUIRED_BRANCH_GROUPS", ["candidate_common", "functional_prior", "brand_prior"]))
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "branch_group" in work.columns:
            filtered = work.loc[work["branch_group"].astype(str).isin(required)]
            if not filtered.empty:
                work = filtered
        if "importance_mean" not in work.columns:
            values.append(_report_missing_value("table7_1_branch_masking_delta", "importance_mean_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [row.get("branch_group"), row.get("user_scope"), row.get("regime")]
                claim_id = "table7_1_branch_masking_delta:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("importance_mean"),
                    ci=_report_ci(row, ["importance_ci_95_low", "ci_low"], ["importance_ci_95_high", "ci_high"]),
                    p=None,
                    n=_report_row_value(row, ["case_count", "fold_count", "n"]),
                    source_file=_report_path_from_maps("branch_contribution"),
                    aggregation="mean NDCG@5 decrease from existing branch masking summary",
                ))
    else:
        values.append(_report_missing_value("table7_1_branch_masking_delta"))
    return values


def _report_gates_19():
    gates = []
    manifest_obj = globals().get("analysis_manifest", {}) if isinstance(globals().get("analysis_manifest", {}), dict) else {}
    checkpoint = globals().get("checkpoint_diagnostics_df")
    checkpoint_values = checkpoint["checkpoint_selection_metric"].dropna().astype(str).unique().tolist() if _report_is_dataframe(checkpoint) and "checkpoint_selection_metric" in checkpoint.columns else None
    observed_checkpoint = {
        "manifest_checkpoint_selection_metric": manifest_obj.get("checkpoint_selection_metric"),
        "checkpoint_diagnostics_values": checkpoint_values,
    }
    checkpoint_bad = manifest_obj.get("checkpoint_selection_metric") not in (None, "validation_ndcg_at_5")
    if checkpoint_values is not None:
        checkpoint_bad = checkpoint_bad or any(value != "validation_ndcg_at_5" for value in checkpoint_values)
    gates.append(_report_gate("checkpoint_selection_metric", observed_checkpoint, "validation_ndcg_at_5", checkpoint_bad))
    manifests = globals().get("interpretation_manifests", {}) if isinstance(globals().get("interpretation_manifests", {}), dict) else {}
    observed_mismatch = {
        "analysis_manifest_disabled_category_mismatch": manifest_obj.get("disabled_category_mismatch"),
        "full_manifest_disabled_category_mismatch": manifests.get("Full", {}).get("disabled_category_mismatch") if isinstance(manifests.get("Full", {}), dict) else None,
    }
    mismatch_bad = any(value is not None and value is not False for value in observed_mismatch.values())
    gates.append(_report_gate("disabled_category_mismatch", observed_mismatch, "false", mismatch_bad))
    return gates


def _report_values_20():
    values = []
    frame = globals().get("scorecard")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            claim_id = "table3_e1_structural_contrast:" + "|".join(str(row.get(column)) for column in ["claim_dimension", "metric"] if column in row)
            values.append(_report_value(
                claim_id=claim_id,
                value=row.get("value"),
                ci=None,
                p=None,
                n=_report_row_value(row, ["n", "case_count", "item_count", "query_count"]),
                source_file=str(_report_output_dir() / "schema_audit_thesis_evidence_scorecard.csv"),
                aggregation="existing schema-audit scorecard value",
            ))
    else:
        values.append(_report_missing_value("table3_e1_structural_contrast_values"))
    return values


def _report_gates_20():
    frame = globals().get("qc_summary")
    return [_report_gate(
        "schema_audit_qc_summary",
        _report_frame_records(frame, limit=50),
        "schema-audit QC rows reported by notebook",
        _report_frame_failed(frame, status_columns=("status",)),
        None if _report_is_dataframe(frame) else "not_available_in_notebook",
    )]


def _report_values_22():
    values = []
    frame = globals().get("paired_summary")
    requested = {
        "Actual_minus_Shuffled",
        "Actual_minus_QCHSfiltered",
        "Actual_minus_QCHS_filtered",
        "QCHS_minus_Actual",
        "Actual_minus_No_User_Brand",
    }
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "population" in work.columns:
            work = work.loc[work["population"].astype(str).eq("non-cold")]
        if "contrast" in work.columns:
            work = work.loc[work["contrast"].astype(str).isin(requested)]
        if work.empty:
            values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas", "requested_non_cold_contrasts_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"table_a_2_prior_policy_delta:{row.get('contrast')}|non-cold",
                    value=row.get("mean_delta"),
                    ci=_report_ci(row, ["ci_low", "bootstrap_ci_95_low"], ["ci_high", "bootstrap_ci_95_high"]),
                    p=_report_row_value(row, ["p", "p_value"]),
                    n=_report_row_value(row, ["n_cases", "n", "case_count"]),
                    source_file=_report_path_from_var("PAIRED_SUMMAR_PATH"),
                    aggregation="existing non-cold paired_summary mean_delta and CI",
                ))
    else:
        values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas"))
    return values


def _report_gates_22():
    gates = []
    paired = globals().get("paired_summary")
    if _report_is_dataframe(paired) and "population" in paired.columns:
        non_cold = paired.loc[paired["population"].astype(str).eq("non-cold")]
        observed_n = _report_frame_records(non_cold, limit=20, columns=["contrast", "population", "n_cases", "n_users"])
    else:
        observed_n = None
    gates.append(_report_gate("non_cold_n", observed_n, "non-cold n recorded in paired_summary", False, None if observed_n is not None else "not_available_in_notebook"))
    model_reuse = globals().get("model_reuse_decision")
    fold_qc = globals().get("fold_qc")
    reuse_observed = {
        "model_reuse_decision": _report_frame_records(model_reuse, limit=20),
        "fold_qc": _report_frame_records(fold_qc, limit=20),
        "canonical_fold_equivalence": globals().get("canonical_fold_equivalence"),
        "canonical_param_equivalence": globals().get("canonical_param_equivalence"),
        "canonical_preprocessing_equivalence": globals().get("canonical_preprocessing_equivalence"),
    }
    gates.append(_report_gate("nb11_fold_hyperparameter_reuse", reuse_observed, "NB11/P2-Q fold and hyperparameter reuse diagnostics are recorded", _report_frame_failed(model_reuse) or _report_frame_failed(fold_qc)))
    seed = globals().get("RANDOM_SEED")
    gates.append(_report_gate("seed", seed, "42", seed not in (None, 42), None if seed is not None else "not_available_in_notebook"))
    return gates


def _report_values_26():
    values = []
    frame = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            values.append(_report_value(
                claim_id=f"section7_1_strong_minus_weak:{row.get('reranker_family')}|{row.get('comparison_id')}",
                value=row.get("strong_minus_weak_delta"),
                ci=_report_ci(row, ["bootstrap_ci_95_lower", "ci_low"], ["bootstrap_ci_95_upper", "ci_high"]),
                p=None,
                n={"strong_case_count": row.get("strong_case_count"), "weak_case_count": row.get("weak_case_count")},
                source_file=_report_path_from_maps("strong_weak_difference_in_delta"),
                aggregation="existing independent strong-minus-weak bootstrap summary",
            ))
    else:
        values.append(_report_missing_value("section7_1_strong_minus_weak_delta_ci"))
    return values


def _report_gates_26():
    gates = []
    coverage = globals().get("regime_pair_coverage_qc")
    gates.append(_report_gate(
        "regime_pair_coverage",
        _report_frame_records(coverage, limit=80),
        "coverage_status PASS for each method x contrast x regime row",
        _report_frame_failed(coverage, status_columns=("coverage_status",)),
        None if _report_is_dataframe(coverage) else "not_available_in_notebook",
    ))
    strong_weak = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(strong_weak):
        observed_overlap = _report_frame_records(strong_weak, limit=30, columns=["reranker_family", "comparison_id", "strong_weak_case_overlap_count", "strong_weak_user_overlap_count"])
        self_flag = False
        for column in ["strong_weak_case_overlap_count", "strong_weak_user_overlap_count"]:
            if column in strong_weak.columns and _report_pd.to_numeric(strong_weak[column], errors="coerce").ne(0).any():
                self_flag = True
    else:
        observed_overlap = None
        self_flag = False
    gates.append(_report_gate("strong_weak_overlap_counts", observed_overlap, "case and user overlap counts equal 0", self_flag, None if observed_overlap is not None else "not_available_in_notebook"))
    return gates


def _report_values_28():
    values = []
    frame = globals().get("table_6_4")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            key = "|".join(str(row.get(column)) for column in ["category_label", "reranker_family", "stage_condition", "candidate_pool_depth"] if column in row)
            values.append(_report_value(
                claim_id=f"table6_4_online_ms_per_query:{key}",
                value=row.get("online_ms_per_query"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="existing online_ms_per_query from table_6_4",
            ))
            values.append(_report_value(
                claim_id=f"table6_4_hardware:{key}",
                value=row.get("compute_device"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="hardware context from existing runtime table/source manifest",
            ))
    else:
        values.append(_report_missing_value("table6_4_method_depth_ms_per_query_and_hardware"))
    return values


def _report_gates_28():
    gates = []
    runtime_qc = globals().get("runtime_qc")
    four_checks = [
        "query_denominator_present",
        "hardware_fields_present",
        "pipeline_stage1_runtime_complete",
        "offline_components_not_in_online_latency",
    ]
    if _report_is_dataframe(runtime_qc) and "check" in runtime_qc.columns:
        raw_four = runtime_qc.loc[runtime_qc["check"].astype(str).isin(four_checks)].copy()
        if raw_four.empty:
            raw_four = runtime_qc.copy()
        observed_qc = _report_frame_records(raw_four, limit=20)
        self_flag = _report_frame_failed(raw_four)
    else:
        observed_qc = None
        self_flag = False
    gates.append(_report_gate("runtime_validation_qc", observed_qc, "four raw runtime validation QC items recorded", self_flag, None if observed_qc is not None else "not_available_in_notebook"))
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    mode = run.get("mode")
    gates.append(_report_gate("aggregation_only", mode, "aggregation_only_no_retrain", mode not in (None, "aggregation_only_no_retrain"), None if mode is not None else "not_available_in_notebook"))
    separation = {
        "online_definition": run.get("online_definition"),
        "offline_excluded": run.get("offline_excluded"),
        "offline_gate": next((row for row in (observed_qc or []) if row.get("check") == "offline_components_not_in_online_latency"), None),
    }
    gates.append(_report_gate("online_offline_separation", separation, "online latency excludes offline components", False if separation["offline_gate"] is not None else True, None if separation["offline_gate"] is not None else "not_available_in_notebook"))
    return gates


def _report_kind():
    name = _report_notebook.lower()
    if name.startswith("15_") or "stage_allocation_summary" in name:
        return "15"
    if name.startswith("18_") or "lightgbm_heldout_interpretation" in name:
        return "18"
    if name.startswith("19_") or "transformer_heldout_interpretation" in name:
        return "19"
    if name.startswith("20_") or "category_schema_audit" in name:
        return "20"
    if name.startswith("22_") or "prior_policy_ablation_lightgbm" in name:
        return "22"
    if name.startswith("26_") or "regime_effect_summary" in name:
        return "26"
    if name.startswith("28_") or "runtime_efficiency_summary" in name:
        return "28"
    return "unknown"


_REPORT_KIND = _report_kind()
_REPORT_SERVED = {
    "15": ["section6_condition_means", "fig6_3"],
    "18": ["fig7_1"],
    "19": ["table7_1", "section7_3"],
    "20": ["table3_e1", "appendix3_e"],
    "22": ["section7_2", "table_a_2"],
    "26": ["section7_1"],
    "28": ["table6_4"],
}.get(_REPORT_KIND, [])
_REPORT_VALUE_BUILDERS = {
    "15": _report_values_15,
    "18": _report_values_18,
    "19": _report_values_19,
    "20": _report_values_20,
    "22": _report_values_22,
    "26": _report_values_26,
    "28": _report_values_28,
}
_REPORT_GATE_BUILDERS = {
    "15": _report_gates_15,
    "18": _report_gates_18,
    "19": _report_gates_19,
    "20": _report_gates_20,
    "22": _report_gates_22,
    "26": _report_gates_26,
    "28": _report_gates_28,
}


def _report_collect_inputs():
    records = []
    seen = set()

    def add_path(path, sha=None):
        if _report_is_missing(path):
            return
        text = str(path).strip()
        if not text:
            return
        candidate = Path(text)
        if _report_is_inside(candidate, _report_output_dir()):
            return
        key = str(candidate)
        if key in seen:
            return
        seen.add(key)
        records.append({"path": key, "sha256": _report_jsonable(sha if sha is not None else _report_file_sha256(candidate, allow_heavy=False))})

    def looks_like_path(text):
        suffix = Path(str(text)).suffix.lower()
        return suffix in {".json", ".csv", ".parquet", ".txt", ".yaml", ".yml", ".ipynb", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}

    def walk(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                key_text = str(key).lower()
                if isinstance(value, (str, Path)) and (key_text.endswith("path") or key_text.endswith("paths") or looks_like_path(value)):
                    sha = None
                    if key_text.endswith("path"):
                        sha = obj.get(str(key).replace("path", "sha256")) or obj.get(str(key).replace("_path", "_sha256"))
                    add_path(value, sha)
                else:
                    walk(value)
        elif isinstance(obj, (list, tuple, set)):
            for value in obj:
                walk(value)
        elif isinstance(obj, (str, Path)) and looks_like_path(obj):
            add_path(obj)

    for obj_name in ["pipeline_manifest", "run_manifest", "analysis_manifest", "manifest"]:
        obj = globals().get(obj_name)
        if isinstance(obj, dict):
            walk(obj)

    inventory = globals().get("runtime_inventory")
    if _report_is_dataframe(inventory):
        for _, row in inventory.iterrows():
            for path_col in ["source_path", "source_manifest", "manifest_path", "path"]:
                if path_col in inventory.columns:
                    sha = None
                    for sha_col in [path_col.replace("path", "sha256"), "source_sha256", "sha256"]:
                        if sha_col in inventory.columns:
                            sha = row.get(sha_col)
                            break
                    add_path(row.get(path_col), sha)

    for name, value in list(globals().items()):
        if not name.endswith("_PATH"):
            continue
        if name.startswith(("OUTPUT", "PREDICTIONS", "PER_CASE", "DELTAS", "PAIRED_SUMMAR", "POPULATION_RESULTS", "PROFILE_DIAGNOSTICS", "SHUFFLE_ASSIGNMENT", "FALLBACK_QC", "FEATURE_CONTRACT", "MODEL_REUSE", "FOLD_QC", "LEAKAGE_QC", "COVERAGE", "RETENTION", "FEATURE_IMPORTANCE", "QC_SUMMAR", "MANIFEST")):
            if _report_is_inside(value, _report_output_dir()):
                continue
        add_path(value)

    return records


_report_run_utc = datetime.now(timezone.utc).isoformat()
_report_values = _REPORT_VALUE_BUILDERS.get(_REPORT_KIND, lambda: [_report_missing_value("requested_values")])()
_report_gates = _REPORT_GATE_BUILDERS.get(_REPORT_KIND, lambda: [_report_gate("requested_gates", None, "not_available_in_notebook", False, "not_available_in_notebook")])()
_report_lineage = {
    "notebook": _report_notebook,
    "category": _report_jsonable(_report_category),
    "run_utc": _report_run_utc,
    "code_sha": _report_file_sha256(_report_nb_path, allow_heavy=True) if _report_nb_path is not None else None,
    "inputs": _report_collect_inputs(),
}


def _report_md_cell(value):
    text = "" if value is None else (_report_jsonable(value))
    if isinstance(text, (dict, list)):
        text = json.dumps(text, ensure_ascii=False, sort_keys=True)
    text = str(text)
    return text.replace("|", "\\|").replace("\n", "<br>")


def _report_markdown(values, gates, lineage):
    lines = []
    lines.append("# Identity & lineage")
    lines.append(f"- notebook: {_report_md_cell(lineage.get('notebook'))}")
    lines.append(f"- category: {_report_md_cell(lineage.get('category'))}")
    lines.append(f"- run_utc: {_report_md_cell(lineage.get('run_utc'))}")
    lines.append(f"- code_sha: {_report_md_cell(lineage.get('code_sha'))}")
    lines.append(f"- input_count: {len(lineage.get('inputs', []))}")
    lines.append("")
    lines.append("# Served thesis elements")
    if _REPORT_SERVED:
        for served_id in _REPORT_SERVED:
            lines.append(f"- {_report_md_cell(served_id)}")
    else:
        lines.append("- not_available_in_notebook")
    lines.append("")
    lines.append("# Computed headline values")
    value_columns = ["claim_id", "value", "ci", "p", "n", "source_file", "aggregation", "thesis_value"]
    lines.append("| " + " | ".join(value_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(value_columns)) + " |")
    for record in values:
        row = dict(record)
        row["thesis_value"] = ""
        lines.append("| " + " | ".join(_report_md_cell(row.get(column)) for column in value_columns) + " |")
    lines.append("")
    lines.append("# QC gates")
    gate_columns = ["gate_id", "observed", "expected_contract", "self_flag"]
    lines.append("| " + " | ".join(gate_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(gate_columns)) + " |")
    for record in gates:
        lines.append("| " + " | ".join(_report_md_cell(record.get(column)) for column in gate_columns) + " |")
    lines.append("")
    lines.append("# Self-detected anomalies")
    anomalies = []
    for record in gates:
        if record.get("self_flag") is True:
            anomalies.append(f"- gate_self_flag: {_report_md_cell(record.get('gate_id'))}")
        if record.get("note"):
            anomalies.append(f"- gate_note: {_report_md_cell(record.get('gate_id'))}: {_report_md_cell(record.get('note'))}")
    for record in values:
        if record.get("note"):
            anomalies.append(f"- value_note: {_report_md_cell(record.get('claim_id'))}: {_report_md_cell(record.get('note'))}")
    lines.extend(anomalies if anomalies else ["- none"])
    lines.append("")
    return "\n".join(lines)


_lineage_path = _report_dir / "lineage.json"
_values_path = _report_dir / "report_values.json"
_gates_path = _report_dir / "qc_gates.json"
_markdown_path = _report_dir / "verification_report.md"
_lineage_path.write_text(json.dumps(_report_lineage, ensure_ascii=False, indent=2), encoding="utf-8")
_values_path.write_text(json.dumps(_report_values, ensure_ascii=False, indent=2), encoding="utf-8")
_gates_path.write_text(json.dumps(_report_gates, ensure_ascii=False, indent=2), encoding="utf-8")
_markdown_path.write_text(_report_markdown(_report_values, _report_gates, _report_lineage), encoding="utf-8")
for _written_path in [_lineage_path, _values_path, _gates_path, _markdown_path]:
    print(_written_path)


/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/lightgbm_interpretation/report/lineage.json
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/lightgbm_interpretation/report/report_values.json
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/lightgbm_interpretation/report/qc_gates.json
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/lightgbm_interpretation/report/verification_report.md
